# Apprentissage par transfert avec un modèle de fondation d'OT pour la segmentation de nappes de pétrole

Ce tutoriel démontre comment utiliser un modèle de fondation d'Observation de la Terre (OT) pré-entraîné globalement pour effectuer une segmentation sémantique sur des données SAR Sentinel-1.

**Énoncé du problème :**  
Étant donné une image SAR Sentinel-1 en rétrodiffusion Sigma0 (polarisations VV et VH), exprimée en unités de décibels (dB), la tâche consiste à déterminer quels pixels de l'image :
- contiennent une nappe de pétrole (*classe = oil, valeur de masque = 1*),
- représentent de l'eau propre (*classe = clean, valeur de masque = 0*) ou contiennent un phénomène ressemblant à une nappe de pétrole (*classe = lookalike, valeur de masque = 0*)

**Approche proposée :**  
Nous construisons un modèle de segmentation sémantique supervisé au-dessus d'un modèle de fondation d'OT pré-entraîné et l'adaptons à cette tâche en l'entraînant à l'aide d'images SAR annotées.

**Choix du modèle :**  
Dans ce tutoriel, nous utilisons le modèle pré-entraîné SATLAS de *Allen Institute*, qui fournit des représentations entraînées globalement pour les données d'observation de la Terre. Voir le dépôt officiel pour les détails sur l'architecture du modèle, les données d'entraînement et les licences :
https://github.com/allenai/satlaspretrain_models et https://huggingface.co/allenai/satlas-pretrain

**Jeu de données :**  
Le jeu de données utilisé dans ce tutoriel est obtenu à partir de Zenodo (DOI: 10.5281/zenodo.8346859). Veuillez vous référer à la source originale (https://zenodo.org/records/8346860) pour les conditions de licence et les restrictions d'utilisation. Notez que les résultats montrés dans ce tutoriel sont obtenus à partir d'un petit sous-ensemble du jeu de données d'entraînement/validation (similaire au tutoriel de classification). Vous pouvez répéter les expériences de ce tutoriel avec le jeu de données complet.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
import OilSpillSegmentation as oilseg

In [ ]:
train_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\train\oil"
train_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\train\lookalike"
train_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\train\clean"
train_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\train\oil"
train_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\train\lookalike"
train_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\train\clean"
train_df = {'images_dir_oil': train_images_dir_oil,
            'images_dir_lookalike': train_images_dir_look,
            'images_dir_clean': train_images_dir_clean,
            'masks_dir_oil': train_masks_dir_oil,
            'masks_dir_lookalike': train_masks_dir_look,
            'masks_dir_clean': train_masks_dir_clean}

val_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\val\oil"
val_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\val\lookalike"
val_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\val\clean"
val_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\val\oil"
val_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\val\lookalike"
val_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\val\clean"
val_df = {'images_dir_oil': val_images_dir_oil,
            'images_dir_lookalike': val_images_dir_look,
            'images_dir_clean': val_images_dir_clean,
            'masks_dir_oil': val_masks_dir_oil,
            'masks_dir_lookalike': val_masks_dir_look,
            'masks_dir_clean': val_masks_dir_clean}

test_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\test\oil"
test_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\test\lookalike"
test_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\test\clean"
test_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\test\oil"
test_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\test\lookalike"
test_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\test\clean"
test_df = {'images_dir_oil': test_images_dir_oil,
          'images_dir_lookalike': test_images_dir_look,
          'images_dir_clean': test_images_dir_clean,
          'masks_dir_oil': test_masks_dir_oil,
          'masks_dir_lookalike': test_masks_dir_look,
          'masks_dir_clean': test_masks_dir_clean}

input_size = 512 # Je choisis de réduire la taille des images mais nous pouvons expérimenter avec les résolutions originales (2048 pixels) aussi.

# Les points de contrôle seront stockés ici.
out_dir_model = r"C:\Users\user\Documents\oilspill\models"
# Les journaux d'exécution (tensorboard) seront stockés ici.
out_dir_runs = r"C:\Users\user\Documents\oilspill\runs_seg"
# Si votre entraînement s'interrompt et que vous souhaitez reprendre l'entraînement.
checkpoint_to_resume = r"C:\user\Documents\oilspill\models\best_seg_model_satlas.pt" 
# Une fois entraîné, ceci devrait représenter le modèle avec le meilleur score F1 sur les données de validation
checkpoint_best = r"C:\Users\user\Documents\oilspill\models\best_seg_model_satlas.pt" 

# Ici je choisis de fusionner les classes clean & lookalike (classification binaire) mais nous pouvons bien sûr expérimenter avec trois classes aussi.

# Paramètres pour la classification binaire
class_names = ["clean","oil"]
num_classes = 2

# # Paramètres pour la classification à trois classes
# class_names = ["clean","oil", "lookalike"]
# num_classes = 3

## Commencer l'entraînement
Pendant l'entraînement, les journaux seront stockés via tensorboard pour le suivi de l'expérience. Utilisez `tensorboard --logdir` pour consulter l'expérience.

La progression sera également sauvegardée sous forme de graphiques.

Les points de contrôle seront sauvegardés après chaque `save_model_every` époques, et le meilleur modèle (basé sur l'IoU de validation) à chaque époque sera également sauvegardé, s'il est atteint.

In [ ]:
run_name = oilseg.main_train(train_df=train_df,
                             val_df=val_df,
                             test_df=test_df,
                             ckpt_dir=out_dir_model,
                             run_dir=out_dir_runs,
                             model_type="satlas",
                             augment_train=True,
                             num_epochs=21,
                             batch_size=16,
                             lr=1e-3,
                             weight_decay=1e-4,
                             freeze_backbone=True,
                             freeze_backbone_partial=True,
                             use_scheduler=True,
                             save_model_every=5,
                             resume_training=False,
                             resume_weights_only=False,
                             resume_ckpt_path=checkpoint_to_resume,
                             input_size=input_size,
                             num_classes=num_classes,
                             class_labels=class_names,
                             dice_factor=0.5,
                             focal_factor=0.5,
                             ignore_index=255)

## Tester la performance

- Les métriques de segmentation du meilleur modèle entraîné seront calculées et affichées.

- Les matrices de confusion seront stockées.

- Les fichiers CSV contenant les métriques par image seront stockés.

In [ ]:
out_dir_plots = os.path.join(out_dir_runs, "plots_" + run_name)
os.makedirs(out_dir_plots, exist_ok=True)
oilseg.main_test(train_df=train_df,
              val_df=val_df,
              test_df=test_df,
              model_path=checkpoint_best,
              plot_out_dir=out_dir_plots,
              model_type="satlas",
              batch_size=16,
              input_size=input_size,
              num_classes=num_classes,
              class_labels=class_names,
              save_seg=False,
              ignore_index=255)

## Inférence
Les sorties de l'inférence incluent :

- Masque de segmentation : Pour chaque image, un masque de segmentation prédit, qui est géoréférencé avec la même transformation/crs que l'image d'entrée, sera sauvegardé.

In [ ]:
oilseg.main_infer(images_dir=test_images_dir_oil, model_path=checkpoint_best,
                      out_masks_dir=os.path.join(out_dir_runs, "infer_oil"), model_type="satlas",
                      batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                      ignore_index=255)
oilseg.main_infer(images_dir=test_images_dir_look, model_path=checkpoint_best,
                  out_masks_dir=os.path.join(out_dir_runs, "infer_lookalike"), model_type="satlas",
                  batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                  ignore_index=255)
oilseg.main_infer(images_dir=test_images_dir_clean, model_path=checkpoint_best,
                  out_masks_dir=os.path.join(out_dir_runs, "infer_clean"), model_type="satlas",
                  batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                  ignore_index=255)

### Voici des exemples de graphiques et de résultats produits par ce modèle.

Exemple de contenu du CSV de prédiction :

    image_path,Precision_clean,Recall_clean,IoU_clean,F1_clean,Dice_clean,Precision_oil,Recall_oil,IoU_oil,F1_oil,Dice_oil,Loss,Accuracy
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00001.tif,0.999,0.998,0.997,0.999,0.999,0.870,0.959,0.839,0.912,0.912,0.049,0.998
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00012.tif,0.991,0.992,0.982,0.991,0.991,0.792,0.773,0.643,0.782,0.782,0.097,0.983
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00018.tif,0.999,0.994,0.994,0.997,0.997,0.840,0.984,0.829,0.906,0.906,0.053,0.994

Exemple de résultat de segmentation :

![](./Assets/sample_segmentation_result.png)


Progression de l'entraînement capturée par Tensorboard :

![](./Assets/train_seg_oil_iou.png)

![](./Assets/train_seg_loss.png)

![](./Assets/val_seg_oil_iou.png)

![](./Assets/val_seg_loss.png)

Évaluation de la performance effectuée via la fonction main_test :

    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES D'ENTRAÎNEMENT
    Métriques de la classe clean
        Précision : 0.995
        Rappel (Recall) : 0.997
        Score F1 : 0.996
        IoU : 0.992
        Score Dice : 0.996
        
    Métriques de la classe oil
        Précision : 0.811
        Rappel (Recall) : 0.720
        Score F1 : 0.763
        IoU : 0.616
        Score Dice : 0.763
    
![](./Assets/CM_SEG_TRAIN.png)
    
    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE VALIDATION
    Métriques de la classe clean
        Précision : 0.998
        Rappel (Recall) : 0.997
        Score F1 : 0.997
        IoU : 0.995
        Score Dice : 0.997
        
    Métriques de la classe oil
        Précision : 0.745
        Rappel (Recall) : 0.752
        Score F1 : 0.748
        IoU : 0.598
        Score Dice : 0.748

![](./Assets/CM_SEG_VALIDATION.png)

    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE TEST
    Métriques de la classe clean
        Précision : 0.975
        Rappel (Recall) : 0.993
        Score F1 : 0.984
        IoU : 0.969
        Score Dice : 0.984
        
    Métriques de la classe oil
        Précision : 0.646
        Rappel (Recall) : 0.329
        Score F1 : 0.436
        IoU : 0.279
        Score Dice : 0.436

![](./Assets/CM_SEG_TEST.png)

Conclusion : Même si nous avons fait de l'apprentissage par transfert à partir d'un modèle fondamental pré-entraîné, il ne semble pas bien généraliser après avoir été affiné sur un très petit jeu de données d'entraînement, en particulier parce que le jeu de données de test présente des différences géographiques considérables avec le jeu de données d'entraînement.

## Exercice 1

1. Visualiser géographiquement les échantillons mal classés et identifier les schémas spatiaux associés aux erreurs.

Comment faire cela ?

- Filtrer les échantillons mal segmentés
- Les tracer sur la carte
- Comparer les emplacements d'erreur avec :
    - les côtes
    - les régions avec une navigation dense
    - les régions absentes des données d'entraînement
- Les diviser par leurs étiquettes d'origine `lookalike` et `clean`, et voir quel pourcentage des erreurs est causé par des confusions avec les cas `lookalike` ?

2. Répéter toute l'expérience avec le jeu de données d'entraînement complet.

## Exercice 2

Comparer les mêmes expériences mais avec un modèle RESNET18-UNET++ pré-entraîné sur le jeu de données ImageNet.
Cette fois, dégeler (*unfreeze*) le backbone car nous savons qu'un backbone gelé ne va pas bien fonctionner.

## Exercice 3

Expérimenter avec la tâche de segmentation différemment :

- Essayer de construire un modèle multi-tâches avec une tête pour la classification et une tête pour la segmentation, puis utiliser les trois classes (y compris `lookalike`) pour les besoins de classification mais seulement deux classes (`oil` vs `clean`/`lookalike`) pour les besoins de segmentation.
- Vérifier si les résultats s'améliorent ; par exemple, le fait d'avoir enseigné au modèle la distinction entre `lookalike` et `oil` a-t-il aidé à améliorer la performance ?